## Configuração da base de dados

Ao analiasr manualmente os dados disponíveis em: 

https://dadosabertos.curitiba.pr.gov.br/conjuntodado/detalhe?chave=b16ead9d-835e-41e8-a4d7-dcc4f2b4b627

observa-se que o banco de dados é cumulativo. Ou seja, o arquivo `2017-02-01_sigesguarda_-_Base_de_Dados` já contém os registros do arquivo `2017-01-01_sigesguarda_-_Base_de_dados`, funcionando como uma mera continuação. Esse padrão se manteve até `2024-02-01_sigesguarda_-_Base_de_Dados.csv`. A partir dessa versão, a coluna `ATENDIMENTO_ANO` foi removida e o esquema do banco de dados foi modificado. 

Diante disso, o objetivo é utilizar o arquivo `2024-02-01_sigesguarda_-_Base_de_Dados.csv` (que abrange dados de 2009 até 12/12/2022) e combiná-lo com a base mais recente disponível: `2026-04-02_sigesguarda_-_Base_de_Dados.csv`. Para isso, é necessário padronizar a base antiga com a base na estrutural atual. As etapas são:

1. Remover a coluna `ATENDIMENTO_ANO` da base antiga;
2. Padronizar a coluna `OCORRENCIA_DATA`. Na base antiga, essa coluna segue o formato `ANO-MÊS-DIA HORA:MINUTO:SEGUNDO.000`. O mesmo tratamento será aplicado à versão mais recente da base, que utiliza o formato `DIA/MÊS/ANO`.

Após essas padronizações, será feita uma busca por registros duplicados, a fim de identificar o ponto correto de mesclagem entre as duas bases.

In [1]:
import sys 
import os 

sys.path.append(os.path.join('..', '..'))

In [2]:
import pandas as pd 
import numpy as np
from pathlib import Path

from config.data_cleaning_config import (
    DEFAULT_DROP_COLUMNS,
    COLUNAS_TEXTO,
    COLUNAS_BINARIAS,
    COLUNA_DIA_SEMANA,
    COLUNA_MES,
    COLUNA_HORA,
    COLUNA_BAIRRO
)

In [3]:
data_dir = Path("../../data/sigesguarda/raw")
arquivo_base_antiga = data_dir / "2024-02-01_sigesguarda_-_Base_de_Dados.csv"
arquivo_base_atual = data_dir / "2026-04-02_sigesguarda_-_Base_de_Dados.csv"

In [4]:
print(data_dir)

../../data/sigesguarda/raw


In [5]:
df_base_antiga = pd.read_csv(arquivo_base_antiga, sep=";", encoding="latin1", dtype=str)

In [6]:
df_base_atual = pd.read_csv(arquivo_base_atual, sep=",", encoding="latin1", dtype=str)

## Base de dados antiga

### 01. Verificação do formato da coluna OCORRENCIA_HORA 

Primeiro, vamos verificar identificar linhas que fogem do padrão observado. Para isso, vamos verificar a coluna `OCORRENCIA_DATA` e identificar os valores que não podem ser interpretados no formato de data definido a seguir:

In [7]:
formato_esperado_base_antiga = "%Y-%m-%d %H:%M:%S.%f"

Em seguida, o código irá selecionar apenas os casos em que `OCORRENCIA_DATA` não está vazia e a conversão falhou. Para esses registros, ele cria uma tabela com duas informações: a linha em que o valor aparece no arquivo original e o valor encontrado naquela linha. 

In [8]:
serie = df_base_antiga["OCORRENCIA_DATA"].astype("string").str.strip()
resultado_validacao = pd.to_datetime(serie, format=formato_esperado_base_antiga, errors="coerce")

invalidos = df_base_antiga.loc[
      serie.notna() & (serie != "") & resultado_validacao.isna(),
      ["OCORRENCIA_DATA"]
  ].copy()

invalidos["linha_ocorrencia"] = invalidos.index + 2
invalidos = invalidos.rename(columns={"OCORRENCIA_DATA": "valor_encontrado"})

display(invalidos[["linha_ocorrencia", "valor_encontrado"]])


,linha_ocorrencia,valor_encontrado
0,2,-----------------------


Esse valor encontrado aparenta ser um dermacador de estilo, que separa o cabeçalho dos dados. Assim, podemos eliminar essa linha fazendo:

In [9]:
df_base_antiga = df_base_antiga.drop(index=0).copy()

### 02. Separando a data em três colunas: dia, mês e ano

Será aplicada uma função chamada `separar_ocorrencia` sobre a base de dados antiga. Essa função utiliza o formato identificado da coluna `OCORRENCIA_DATA` e extrai três novas colunas: 

- `OCORRENCIA_ANO`; 
- `OCORRENCIA_MES`; 
- `OCORRENCIA_DIA`. 

In [10]:
def separar_ocorrencia_data(df, format):
    if "OCORRENCIA_DATA" not in df.columns:
        return df 
    
    df = df.copy()
    serie = df["OCORRENCIA_DATA"].astype("string").str.strip()

    datas = pd.to_datetime(
        serie,
        format=format,
        errors="coerce"
    )

    df = df.loc[datas.notna()].copy()
    datas = datas.loc[datas.notna()]

    df["OCORRENCIA_ANO"] = datas.dt.year.astype("Int64")
    df["OCORRENCIA_MES"] = datas.dt.month.astype("Int64")
    df["OCORRENCIA_DIA"] = datas.dt.day.astype("Int64")

    df = df.drop(columns=["OCORRENCIA_DATA"])

    return df

In [11]:
df_base_antiga = separar_ocorrencia_data(df_base_antiga, format=formato_esperado_base_antiga)

In [12]:
df_base_antiga[["OCORRENCIA_ANO", "OCORRENCIA_MES", "OCORRENCIA_DIA"]].head(10)

,OCORRENCIA_ANO,OCORRENCIA_MES,OCORRENCIA_DIA
1,2009,1,1
2,2009,1,1
3,2009,1,1
4,2009,1,1
5,2009,1,1
6,2009,1,1
7,2009,1,1
8,2009,1,1
9,2009,1,1
10,2009,1,1


Além disso, a base de dados antiga não possui a coluna `ATENDIMENTO_ANO`, que é uma informação redundante com `OCORRENCIA_ANO`. Assim, podemos descarta-la:

In [13]:
df_base_antiga = df_base_antiga.drop(columns=["ATENDIMENTO_ANO"], errors="ignore")

## Base de dados atual

### 01. Verificação do formato da coluna OCORRENCIA_HORA 

Na base de dados nova, a coluna `OCORRENCIA_HORA` está no formato:

In [14]:
formato_esperado_base_nova = "%d/%m/%Y"

In [15]:
serie = df_base_atual["OCORRENCIA_DATA"].astype("string").str.strip()
resultado_validacao = pd.to_datetime(serie, format=formato_esperado_base_nova, errors="coerce")

invalidos = df_base_atual.loc[
      serie.notna() & (serie != "") & resultado_validacao.isna(),
      ["OCORRENCIA_DATA"]
  ].copy()

invalidos["linha_ocorrencia"] = invalidos.index + 2
invalidos = invalidos.rename(columns={"OCORRENCIA_DATA": "valor_encontrado"})

display(invalidos[["linha_ocorrencia", "valor_encontrado"]])


,linha_ocorrencia,valor_encontrado


### 02. Separando a data em três colunas: dia, mês e ano

In [16]:
df_base_atual = separar_ocorrencia_data(df_base_atual, format=formato_esperado_base_nova)

In [17]:
df_base_atual[["OCORRENCIA_ANO", "OCORRENCIA_MES", "OCORRENCIA_DIA"]].head(10)

,OCORRENCIA_ANO,OCORRENCIA_MES,OCORRENCIA_DIA
0,2022,11,24
1,2022,12,7
2,2022,12,8
3,2022,12,8
4,2022,12,12
5,2022,12,12
6,2022,12,12
7,2022,12,12
8,2022,12,12
9,2022,12,12


## Analisando se há convergência entre as duas bases de dados

Como ambas as bases de dados contêm registros referentes a novembro de 2022, será feita uma análise para verificar a existência de duplicidade nesse mês. 

Para isso, serão extraídos, tanto da base antiga quanto da atual, apenas os registros correspondentes a novembro de 2022. A identificação de duplicatas será baseada na coluna `OCORRENCIA_CODIGO`.

In [18]:
for df in (df_base_antiga, df_base_atual):
    df["OCORRENCIA_CODIGO"].astype("string").str.strip()

    recorte_antiga = df_base_antiga.loc[
        (df_base_antiga["OCORRENCIA_ANO"] == 2022) &
        (df_base_antiga["OCORRENCIA_MES"] == 11)
    ].copy()

    recorte_atual = df_base_atual.loc[
        (df_base_atual["OCORRENCIA_ANO"] == 2022) &
        (df_base_antiga["OCORRENCIA_MES"] == 11)
    ].copy()

    codigos_em_comum = sorted(
        set(recorte_antiga["OCORRENCIA_CODIGO"].dropna()) &
        set(recorte_atual["OCORRENCIA_CODIGO"].dropna())
    )

    print(f"Ocorrências presentes nas duas bases: {len(codigos_em_comum)}")

Ocorrências presentes nas duas bases: 0
Ocorrências presentes nas duas bases: 0


Como não foram identificados dados duplicados, vamos mergear as duas bases de dados.

In [19]:
df_final = pd.concat([df_base_antiga, df_base_atual], ignore_index=True)

In [20]:
df_final.head()

,ATENDIMENTO_BAIRRO_NOME,EQUIPAMENTO_URBANO_NOME,FLAG_EQUIPAMENTO_URBANO,FLAG_FLAGRANTE,LOGRADOURO_NOME,NATUREZA1_DEFESA_CIVIL,NATUREZA1_DESCRICAO,NATUREZA2_DEFESA_CIVIL,NATUREZA2_DESCRICAO,NATUREZA3_DEFESA_CIVIL,...,OCORRENCIA_MES,OPERACAO_DESCRICAO,ORIGEM_CHAMADO_DESCRICAO,REGIONAL_FATO_NOME,SECRETARIA_NOME,SECRETARIA_SIGLA,SERVICO_NOME,SITUACAO_EQUIPE_DESCRICAO,NUMERO_PROTOCOLO_156,OCORRENCIA_DIA
0,CIDADE INDUSTRIAL,NaN,NÃO,NÃO,DAVI XAVIER DA SILVA,0,Alarmes,NaN,NaN,NaN,...,1,NaN,.,CIC,FUNDAÇÃO DE AÇÃO SOCIAL,FAS,SIGA,NaN,NaN,1
1,FAZENDINHA,BOSQUE DA FAZENDINHA,SIM,NÃO,CARLOS KLEMTZ,0,Roubo,NaN,NaN,NaN,...,1,NaN,153,PORTÃO,SECRETARIA MUNICIPAL MEIO AMBIENTE,SMMA,NORMAL,NaN,NaN,1
2,UBERABA,NaN,NÃO,NÃO,DOUTOR JOÃO DE PAULA MOURA BRITO,0,Animais,NaN,NaN,NaN,...,1,NaN,156,CAJURU,SECRETARIA MUNICIPAL DA SAÚDE,SMS,SACAF,NaN,2640856,1
3,SÍTIO CERCADO,NaN,NÃO,NÃO,EDGARD CAVALCANTI DE ALBUQUERQUE,0,Animais,NaN,NaN,NaN,...,1,NaN,156,BAIRRO NOVO,SECRETARIA MUNICIPAL DA SAÚDE,SMS,SACAF,NaN,2640854,1
4,TATUQUARA,CENTRO DE ESPORTE E LAZER SANTA RITA,SIM,NÃO,CARLOS MUNHOZ DA ROCHA,0,Alarmes,NaN,NaN,NaN,...,1,NaN,.,PINHEIRINHO,FUNDAÇÃO DE AÇÃO SOCIAL,FAS,SIGA,NaN,NaN,1


In [21]:
output_dir = Path("../../data/sigesguarda/base_de_dados")
output_dir.mkdir(parents=True, exist_ok=True)

In [22]:
arquivo_final = output_dir / "base_unificada.csv"

In [23]:
df_final.to_csv(arquivo_final, sep=";", encoding="latin1", index=False)